In [13]:
import torch
import torch.nn as nn
import math
import pandas as pd
import numpy as np

In [30]:
with open('../data/input.txt', 'r')as f:
    text = f.read()

In [31]:
text[:500]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor"

In [32]:
len(set(text))

65

In [33]:
stoi={}
itos={}

In [35]:
i=0

In [36]:
for char in text:
    if char not in stoi:
        stoi[char]=i
        itos[i]=char
        i+=1

In [37]:
len(stoi)

65

In [42]:
stoi['n']

9

In [43]:
itos[9]

'n'

In [40]:
stoi['\n']

11

In [41]:
itos[11]

'\n'

In [44]:
encoded = torch.tensor([stoi[char] for char in text])

In [46]:
decoded = ''.join(itos[i.item()] for i in encoded[:100])

In [47]:
decoded

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [48]:
text[0:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [50]:
encoded[:100]

tensor([ 0,  1,  2,  3,  4,  5,  6,  1,  4,  1,  7,  8,  9, 10, 11, 12,  8, 13,
        14,  2,  8,  5, 15,  8,  5, 16,  2, 14, 17,  8,  8, 18,  5, 19,  9, 20,
         5, 13, 21,  2,  4, 22,  8,  2, 23,  5, 22,  8, 19,  2,  5, 24,  8,  5,
         3, 16,  8, 19, 25, 26, 11, 11, 27, 28, 28, 10, 11, 29, 16,  8, 19, 25,
        23,  5,  3, 16,  8, 19, 25, 26, 11, 11,  0,  1,  2,  3,  4,  5,  6,  1,
         4,  1,  7,  8,  9, 10, 11, 30, 14, 21])

In [52]:
train = encoded[:int((0.9)*len(text))]
test =  encoded[int((0.9)*len(text)):]

In [53]:
len(train)

1003854

In [55]:
len(test)

111540

In [63]:
batchsize=4
blocksize=8
torch.manual_seed(1337)

In [71]:
def make_batch():
    indexx = torch.randint(len(train) - blocksize,(batchsize,))
    x = torch.stack([train[i:i+blocksize] for i in indexx])
    y = torch.stack([train[i+1:i+blocksize+1] for i in indexx])
    return x,y

In [72]:
xb,yb = make_batch()

In [73]:
xb.shape

torch.Size([4, 8])

In [75]:
yb.shape

torch.Size([4, 8])

In [76]:
class casualMHAttention(nn.Module):
    def __init__(self, d_model, num_heads, blocksize):
        super(casualMHAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.blocksize = blocksize
        self.query = nn.Linear(d_model,d_model)
        self.key = nn.Linear(d_model,d_model)
        self.value = nn.Linear(d_model,d_model)
        self.output = nn.Linear(d_model,d_model)
        mask = torch.tril(torch.ones(blocksize,blocksize))
        self.register_buffer('mask', mask.view((1,1,blocksize,blocksize)))
        self.headsize = d_model//num_heads

    def split_heads(self, x):
        batchsize,seq_len,d_model = x.size()
        return x.view(batchsize,seq_len,self.num_heads,self.headsize).transpose(1,2)

    def forward(self,x):
        batchsize,seq_len,d_model = x.size()
        q= self.split_heads(self.query(x))
        k= self.split_heads(self.key(x))
        v= self.split_heads(self.value(x))
        att = torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.headsize)
        att = att.masked_fill(self.mask[:,:,:seq_len,:seq_len]==0, float('-inf'))
        att = torch.softmax(att,dim=-1)@v
        att = att.transpose(1,2).contiguous().view(batchsize,seq_len,d_model)
        output = self.output(att)
        return output

In [77]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model):
        super(FeedForwardNetwork, self).__init__()
        self.dff = 4*d_model
        self.d_model = d_model
        self.layer1 = nn.Linear(self.d_model,self.dff)
        self.layer2 = nn.Linear(self.dff,self.d_model)
    def forward(self,x):
        x = nn.functional.gelu(self.layer1(x))
        x = self.layer2(x)
        return x

In [79]:
class PreLNResidualBlock(nn.Module):
    def __init__(self, d_model, sub_layer):
        super(PreLNResidualBlock, self).__init__()
        self.sub_layer = sub_layer
        self.d_model = d_model
        self.norm = nn.LayerNorm(d_model)
    def forward(self,x):
        normalized_x =self.norm(x)
        sub_layer_out = self.sub_layer(normalized_x)
        return x +sub_layer_out

In [102]:
class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, blocksize):
        super(PositionalEmbedding, self).__init__()
        self.d_model = d_model
        self.emb = nn.Embedding(blocksize, d_model)
    def forward(self,x):
        seqlen = x.size(1)
        pos = torch.arange(0,seqlen,dtype=torch.long, device=x.device)
        return x + self.emb(pos)

In [103]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, blocksize):
        super(Transformer, self).__init__()
        self.ff = PreLNResidualBlock(d_model, FeedForwardNetwork(d_model))
        self.att = PreLNResidualBlock(d_model, casualMHAttention(d_model, num_heads, blocksize))

    def forward(self, x):
        x = self.att(x)
        x = self.ff(x)
        return x

In [104]:
class LanguageModel(nn.Module):
    def __init__(self, d_model, num_heads, blocksize, vocab_size, num_layers):
        super(LanguageModel, self).__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = PositionalEmbedding(d_model, blocksize)
        self.transformers = nn.Sequential(*[Transformer(d_model, num_heads, blocksize) for _ in range(num_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_emb.weight

    def forward(self, x):
        x = self.token_emb(x)
        x=self.pos_emb(x)

        x=self.transformers(x)
        x = self.ln(x)
        logits = self.head(x)
        return logits

In [105]:
# for unit testing

blocksize=8
batchsize=4
d_model = 64
num_heads= 8

In [117]:
x = torch.randn(batchsize,blocksize,d_model)

In [118]:
att = casualMHAttention(d_model, num_heads, blocksize)
out = att(x)
assert out.shape == (batchsize,blocksize, d_model)
print('okay')

okay


In [119]:
ffn = FeedForwardNetwork(d_model)
out = ffn(x)
assert out.shape==x.shape
print('okay')

okay


In [120]:
transformer_block = Transformer(d_model, num_heads, blocksize)
out = transformer_block(x)
assert out.shape==x.shape
print('okay')

okay


In [121]:
model = LanguageModel(d_model, num_heads, blocksize,len(set(text)),num_layers=6)
x,y=make_batch()
logits = model(x)
assert logits.shape==(batchsize,blocksize, len(set(text)))
print('okay')

okay


In [129]:
# to verify causal compliance
xtest = torch.randn(batchsize,blocksize,d_model,requires_grad=True)
attlayer = casualMHAttention(d_model, num_heads, blocksize)
out = attlayer(xtest)

In [130]:
t = 3
loss_at_t = out[0,t,:].sum()

In [131]:
loss_at_t.backward()

In [132]:
future_grads = xtest.grad[0,t+1:,:].abs().sum().item()

In [134]:
assert future_grads==0
print('okay')

okay


In [143]:
# overfitting a single batch
model = LanguageModel(d_model, num_heads, blocksize,len(set(text)),num_layers=6)
x,y=make_batch()

optim = torch.optim.Adam(model.parameters(), lr=0.01)
epochs=250
for i in range(epochs):
    logits = model(x)
    b,t,c = logits.shape
    logitsflat = logits.view(b*t,c)
    tarflat = y.view(b*t)
    loss = torch.nn.functional.cross_entropy(logitsflat, tarflat)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    optim.step()
    if i%10==0:
        print(f'epoch: {i} loss: {loss.item():.10f}')



epoch: 0 loss: 37.4099769592
epoch: 10 loss: 0.1330094486
epoch: 20 loss: 0.0511866026
epoch: 30 loss: 0.0488361716
epoch: 40 loss: 0.0446215644
epoch: 50 loss: 0.0435519367
epoch: 60 loss: 0.0436716601
epoch: 70 loss: 0.0434496254
epoch: 80 loss: 0.0434489623
epoch: 90 loss: 0.0434221178
epoch: 100 loss: 0.0434055403
epoch: 110 loss: 0.0433953367
epoch: 120 loss: 0.0433876589
epoch: 130 loss: 0.0433819480
epoch: 140 loss: 0.0433771908
epoch: 150 loss: 0.0433729663
epoch: 160 loss: 0.0433693118
epoch: 170 loss: 0.0433660559
epoch: 180 loss: 0.0433631875
epoch: 190 loss: 0.0433606021
epoch: 200 loss: 0.0433582701
epoch: 210 loss: 0.0433561578
epoch: 220 loss: 0.0433542356
epoch: 230 loss: 0.0433525071
epoch: 240 loss: 0.0433508791
